# Tool-trace generation (M1)

Runs generated seeds through **Qwen3-4B** as the teacher, executes each tool call
for real inside a disposable sandbox, records the trace, and harvests it through
the six gates.

This is the runner the harvester expects. It is not the harness's own loop — the
trace contract is the same, and swapping in `agentic-loop` later changes who
executes the call, not the format.

**Seeds are generated, never harvested.** `agentic-loop dataset export` is
metadata-only because real task text and paths must never be published; nothing
here reads a real run.


In [ ]:
import glob, json, os, sys
from pathlib import Path

tools_dir = Path(sorted(glob.glob('/kaggle/input/**/tool_traces.py', recursive=True))[0]).parent
sys.path.insert(0, str(tools_dir))
from tool_traces import make_seeds, harvest, TOOLS
print('tools from', tools_dir)

# tools/kaggle_lab.py substitutes __N_SEEDS__; a bare push keeps 200.
N_SEEDS = '__N_SEEDS__'
if N_SEEDS.startswith('__'):
    N_SEEDS = '200'
N_SEEDS = int(N_SEEDS)
SEEDS = Path('/kaggle/working/seeds')
make_seeds(SEEDS, N_SEEDS, None)
seeds = [json.loads(l) for l in (SEEDS / 'seeds.jsonl').read_text().splitlines()]
print('seeds:', len(seeds))
print('tool surface:', list(TOOLS))


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

TEACHER = 'Qwen/Qwen3-4B'
tok = AutoTokenizer.from_pretrained(TEACHER)
model = AutoModelForCausalLM.from_pretrained(TEACHER, torch_dtype=torch.float16,
                                             device_map={'': 0}).eval()
print('teacher on', next(model.parameters()).device)


In [ ]:
import re, shlex, subprocess

SYSTEM = ("You are a tool-using agent. Reply with exactly one JSON object. "
          "To call a tool: {\"tool\": <name>, \"args\": {...}}. "
          "To finish: {\"done\": true, \"answer\": <answer>}. "
          f"Tools: {json.dumps(TOOLS)}")

ALLOWED_CMD = {'wc', 'ls', 'cat', 'head'}

def confined(sandbox: Path, rel: str) -> Path:
    p = (sandbox / rel).resolve()
    if not str(p).startswith(str(sandbox.resolve())):
        raise ValueError('path escapes sandbox')
    return p

def execute(sandbox: Path, name: str, args: dict) -> dict:
    try:
        if name == 'read_file':
            return {'content': confined(sandbox, args['path']).read_text(errors='replace')[:4000]}
        if name == 'list_dir':
            d = confined(sandbox, args.get('path', '.'))
            return {'content': '\n'.join(sorted(p.name for p in d.iterdir()))}
        if name == 'grep':
            pat = args['pattern']; d = confined(sandbox, args.get('path', '.'))
            hits = []
            for f in d.rglob('*') if d.is_dir() else [d]:
                if f.is_file():
                    for i, line in enumerate(f.read_text(errors='replace').splitlines(), 1):
                        if pat in line:
                            hits.append(f'{f.relative_to(sandbox)}:{i}:{line[:200]}')
            return {'content': '\n'.join(hits[:40]) or '(no matches)'}
        if name == 'run_command':
            argv = shlex.split(args['cmd'])
            if not argv or argv[0] not in ALLOWED_CMD:
                return {'content': '', 'error': f'command {argv[0] if argv else ""} not allowed'}
            r = subprocess.run(argv, cwd=sandbox, capture_output=True, text=True, timeout=20)
            return {'content': (r.stdout + r.stderr)[:2000]}
        if name == 'write_file':
            p = confined(sandbox, args['path']); p.parent.mkdir(parents=True, exist_ok=True)
            p.write_text(args['content'])
            return {'content': f'wrote {len(args["content"])} bytes'}
        return {'content': '', 'error': f'unknown tool {name}'}
    except Exception as e:
        return {'content': '', 'error': str(e)[:200]}

def ask(messages, max_new=200):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        out = model.generate(**ids, max_new_tokens=max_new, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)

def parse(text):
    for m in re.finditer(r'\{.*\}', text, re.S):
        try:
            return json.loads(m.group(0))
        except Exception:
            continue
    return None

runs_path = Path('/kaggle/working/runs.jsonl')
done = 0
with runs_path.open('w') as out:
    for seed in seeds:
        sandbox = Path(seed['sandbox'])
        msgs = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': seed['task']}]
        calls, results, answer, outcome = [], [], '', None
        for step in range(4):
            raw = ask(msgs)
            obj = parse(raw)
            if obj is None:
                msgs.append({'role': 'assistant', 'content': raw})
                msgs.append({'role': 'user', 'content': 'Reply with one JSON object only.'})
                continue
            if 'done' in obj:
                answer = str(obj.get('answer', ''))
                outcome = 'done' if answer else None
                break
            name, args = obj.get('tool'), obj.get('args', {})
            res = execute(sandbox, name, args)
            calls.append({'name': name, 'args': args})
            results.append(res)
            msgs.append({'role': 'assistant', 'content': json.dumps(obj)})
            msgs.append({'role': 'user', 'content': 'TOOL RESULT: ' + json.dumps(res)[:3000]})
        out.write(json.dumps({'seed': seed['seed'], 'task': seed['task'], 'system': SYSTEM,
                              'calls': calls, 'results': results, 'answer': answer,
                              'outcome': outcome}) + '\n')
        done += 1
        if done % 25 == 0:
            print(done, 'runs')
print('runs:', done)


In [ ]:
out = Path('/kaggle/working/traces.jsonl')
try:
    harvest(Path('/kaggle/working/runs.jsonl'), out)
except SystemExit as e:
    print('harvest stopped:', e)
kept = [json.loads(l) for l in out.read_text().splitlines()] if out.exists() else []
print('kept traces:', len(kept))
if kept:
    from collections import Counter
    print('tools used:', Counter(c['name'] for t in kept for c in t['calls']).most_common())
    print('example answer:', kept[0]['answer'][:200])
